In [23]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

from sklearn.decomposition import PCA
from numpy.linalg import eig
from numpy import array, mean, cov

In [20]:
df, target = load_iris(return_X_y=True)
df = pd.DataFrame(df, columns=["sepal length (cm)","sepal width (cm)","petal length (cm)","petal width (cm)"])
df["label"] = target
df_features = df.drop(columns=["label"])
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),label
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [45]:
x, y = load_iris(return_X_y=True)



def evaluate_model(model_func, model_name, features, df_x=df_features, df_y=df["label"]):
    #gnb = GaussianNB()
    model = model_func()
    x_train, x_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.5, random_state=0)
    y_pred1 = model.fit(x_train, y_train).predict(x_test)

    x_train, x_test = x_test, x_train
    y_train, y_test = y_test, y_train
    #gnb = GaussianNB()
    model = model_func()
    y_pred2 = model.fit(x_train, y_train).predict(x_test)

    y_pred = np.round(np.append(y_pred2, y_pred1))
    y_true = np.append(y_test, y_train)
    print(f"Evaluating {model_name} model")
    print(f"Accuracy: {round((y_pred == y_true).sum() / len(y), 4)}")
    print(f"Confusion Matrix:\n {confusion_matrix(y_true, y_pred, labels=[0, 1, 2])}")
    print(f"Features used: {', '.join(features)}")

In [46]:
# Part 1: original 4 features.
evaluate_model(lambda : DecisionTreeClassifier(), "Decision Tree", df_features.columns)

Evaluating Decision Tree model
Accuracy: 0.96
Confusion Matrix:
 [[50  0  0]
 [ 0 47  3]
 [ 0  3 47]]
Features used: sepal length (cm), sepal width (cm), petal length (cm), petal width (cm)


In [49]:
# Part 2: PCA
features = ["p0", "p1", "p2", "p3"]
df_features = df.drop(columns=["label"])
pca = PCA(n_components=4)
df_pca = pd.DataFrame(pca.fit_transform(df_features), columns=features)

A = df_pca.values
M = mean(A.T, axis=1)
C = A - M
V = cov(C.T)
eigen_values, eigen_vectors = eig(V)
print(f"Eigen values: {eigen_values}, \nEigen vectors:\n{eigen_vectors}")
eigen_sum = sum(eigen_values)
povs = [round(v * 100 / eigen_sum, 2) for v in np.cumsum(eigen_values)]
povs_print = [f'{round(v * 100 / eigen_sum, 2)}%' for v in np.cumsum(eigen_values)]
print(f"PoVs: {', '.join(povs_print)}")

max_idx = len([f for idx, f in enumerate(features) if povs[idx] < 90])
feature_subset = features[:max_idx + 1]
print(f"Feature subset that makes PoV > 0.9: {', '.join(feature_subset)}")
evaluate_model(lambda : DecisionTreeClassifier(), "Decision Tree", feature_subset, df_x=df_pca[feature_subset])

Eigen values: [4.22824171 0.24267075 0.0782095  0.02383509], 
Eigen vectors:
[[ 1.00000000e+00  3.90334171e-18 -1.41434979e-17 -5.61420103e-18]
 [ 0.00000000e+00  1.00000000e+00 -1.85242795e-16 -2.28950853e-16]
 [ 0.00000000e+00 -4.34114550e-16  1.00000000e+00 -2.40311063e-16]
 [ 0.00000000e+00 -4.81688688e-17 -1.14567849e-16  1.00000000e+00]]
PoVs: 92.46%, 97.77%, 99.48%, 100.0%
Feature subset that makes PoV > 0.9: p0
Evaluating Decision Tree model
Accuracy: 0.9133
Confusion Matrix:
 [[50  0  0]
 [ 0 44  6]
 [ 0  7 43]]
Features used: p0


In [50]:
# Part 3: Simulated Annealing

